# A worked example: from a MEANtools correlation to a PlantMetWiki pathway, tier by tier

**Question.** MEANtools reports gene-metabolite correlations. When can we say that such a correlation points to a pathway of [PlantMetWiki](https://plantmetwiki.bioinformatics.nl), and how much do we trust the link? The paired-omics pilot found **0 of 721 fully mapped associations that share a PlantMetWiki pathway** (see [`pairedomics_overlap`](../../pilots/pairedomics_overlap/)). Here we take one association that this direct mapping **misses** and climb the evidence tiers (locus, UniProt, EC, orthology, sequence similarity, co-expression, chemical similarity, InChIKey, Rhea) to see **at which level the pathway can be retrieved**, and what each failure means.

**The example.** Tomato leaves treated with the fungus *Cladosporium fulvum* (MEANtools case study, Singh et al. 2025). The genes `Solyc08g068730` (THT1-3) and `Solyc08g068710` code for putative **tyramine hydroxycinnamoyl transferases (THT)**, the enzymes of the PlantMetWiki pathway **hydroxycinnamic acid tyramine amides biosynthesis (PlantCyc PWY-5474)**. They correlate strongly with a feature of m/z 300.1225 that fits **caffeoyltyramine**, a compound that **is not in PlantMetWiki**. This is known biology, so we can check whether the tiers find it, and read the failures.

**Structure.** 1 the correlations, 2 which compound the feature is, 3 gene side, 4 metabolite side, 5 reaction side, 6 summary and score, 7 how to read MEANtools output.

*Needs:* `pilots/pairedomics_overlap/data/Jeon_fungal.sqlite` (downloaded by `match_meantools.py` of that pilot), internet (UniProt, OrthoDB, Rhea and PlantMetWiki are queried live), RDKit and Biopython (`requirements.txt`).

In [1]:
from pathlib import Path
import sqlite3, io, re, warnings
import numpy as np, pandas as pd, requests
warnings.filterwarnings('ignore')
pd.set_option('display.width', 220); pd.set_option('display.max_colwidth', 70); pd.set_option('display.max_rows', 60)

PILOT = Path('../../pilots/pairedomics_overlap'); MET = Path('../../pilots/metabolomics_overlap/data')
DB = PILOT / 'data' / 'Jeon_fungal.sqlite'
assert DB.exists(), 'run pilots/pairedomics_overlap/match_meantools.py once: it downloads the MEANtools database (Zenodo 15697913)'
con = sqlite3.connect(DB)
PLANTCYC = 'PWY-5474'    # hydroxycinnamic acid tyramine amides biosynthesis
# the two tomato THT-like genes and their UniProt entries
GENES = {'Solyc08g068730': 'A0A3Q7HPP9',    # 'THT1-3'
         'Solyc08g068710': 'A0A3Q7HQ14'}    # annotated only as 'N-acetyltransferase domain-containing protein'

## The story in one picture

The figure below shows how one MEANtools association (left) reaches a PlantMetWiki pathway (right) **only through indirect tiers**. Solid green arrows are tiers that work; dashed red arrows are the direct tiers that fail; the dotted grey arrow is an analogy. Everything in it is computed in the sections below.

In [2]:
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch, Patch
from matplotlib.lines import Line2D
INK, MUTED, LIGHT = '#1f2933', '#6b7280', '#e5e7eb'
BLUE, ORANGE, GREEN, RED, GREY = '#2a6f97', '#d9822b', '#3f8f5a', '#b5473f', '#9aa3ad'
plt.rcParams.update({'font.size': 9, 'axes.edgecolor': MUTED, 'axes.labelcolor': INK, 'xtick.color': INK, 'ytick.color': INK, 'text.color': INK,
                     'axes.spines.top': False, 'axes.spines.right': False, 'figure.dpi': 110})
def save(fig, name): fig.savefig(f'figures/{name}.png', dpi=170, bbox_inches='tight')

KIND = {'data': ('#dbe9f4', BLUE), 'kg': ('#dff0e4', GREEN), 'ext': ('#fbead7', ORANGE), 'hyp': ('#f3f4f6', MUTED)}
fig, ax = plt.subplots(figsize=(12.6, 7.4)); ax.set_xlim(0, 12.6); ax.set_ylim(0, 7.4); ax.axis('off')
N = {}
def node(key, x, y, w, h, text, kind, dashed=False, fs=8.6):
    fc, ec = KIND[kind]
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.02,rounding_size=0.12', fc=fc, ec=ec, lw=1.3, ls='--' if dashed else '-'))
    ax.text(x + w / 2, y + h / 2, text, ha='center', va='center', fontsize=fs)
    N[key] = (x, y, w, h)
def pt(key, side, f=0.5):
    x, y, w, h = N[key]
    return {'l': (x, y + h * f), 'r': (x + w, y + h * f), 't': (x + w * f, y + h), 'b': (x + w * f, y)}[side]
def edge(a, sa, b, sb, label, style, lp, fa=0.5, fb=0.5, rad=0.0):
    col, ls, lw = {'ok': (GREEN, '-', 2.0), 'weak': (MUTED, (0, (1, 2)), 1.7), 'corr': (BLUE, '-', 3.2)}[style]
    (x1, y1), (x2, y2) = pt(a, sa, fa), pt(b, sb, fb)
    ax.add_patch(FancyArrowPatch((x1, y1), (x2, y2), arrowstyle='-' if style == 'corr' else '-|>', mutation_scale=13, color=col, ls=ls, lw=lw, connectionstyle=f'arc3,rad={rad}', shrinkA=2, shrinkB=2))
    if label: ax.text(lp[0], lp[1], label, ha='center', va='center', fontsize=7.8, color=col if style != 'weak' else MUTED, bbox=dict(fc='white', ec='none', pad=1.0, alpha=0.96))
def fail(x, y, text, ha='left'):
    ax.text(x, y, '✗ ' + text, ha=ha, va='center', fontsize=7.9, color=RED, bbox=dict(fc='#fbeceb', ec=RED, lw=0.8, pad=2.2, boxstyle='round,pad=0.25'))

ax.text(1.5, 7.1, 'MEANtools data (tomato)', ha='center', fontweight='bold', color=BLUE)
ax.text(6.0, 7.1, 'external databases and a hypothesis', ha='center', fontweight='bold', color=ORANGE)
ax.text(11.0, 7.1, 'PlantMetWiki (knowledge graph)', ha='center', fontweight='bold', color=GREEN)
node('g', 0.1, 5.2, 2.9, 1.3, 'genes\nSolyc08g068730 (THT1-3)\nSolyc08g068710', 'data')
node('f', 0.1, 0.9, 2.9, 1.2, 'feature M300T887\nm/z 300.1225, RT 887 s\n(+ M322T887, M301T887)', 'data')
node('og', 4.3, 6.1, 3.4, 0.8, 'OrthoDB group 42360at4069\n(potato and tobacco THT)', 'ext')
node('seq', 4.3, 4.85, 3.4, 0.9, 'UniProt THT enzymes, EC 2.3.1.110\n(potato, tobacco): 73-89 % identical', 'ext')
node('rhea', 4.3, 3.15, 3.4, 0.8, 'Rhea: 4 reactions with EC 2.3.1.110', 'ext')
node('hyp', 4.3, 0.9, 3.4, 1.2, 'caffeoyltyramine\n(hypothesis, not in PlantMetWiki)', 'hyp', dashed=True)
node('ec', 9.2, 5.5, 3.3, 1.0, 'EC 2.3.1.110 on 4 reactions\n(no Rhea id, no UniProt)', 'kg')
node('pw', 9.2, 3.0, 3.3, 1.4, 'pathway PWY-5474 (PC536)\nhydroxycinnamic acid\ntyramine amides biosynthesis', 'kg', fs=9)
node('am', 9.2, 0.9, 3.3, 1.2, '4-coumaroyl-, feruloyl-,\nsinapoyl-, cinnamoyl-tyramine\n(in the KG)', 'kg')
# correlation
edge('g', 'b', 'f', 't', 'MEANtools\nr 0.88, MR 7.6', 'corr', (1.95, 3.55), fa=0.5, fb=0.5)
# gene side
edge('g', 'r', 'og', 'l', 'ortholog\n(one gene)', 'ok', (3.65, 6.72), fa=0.85, fb=0.5, rad=-0.08)
edge('g', 'r', 'seq', 'l', 'sequence\nsimilarity', 'ok', (3.65, 5.12), fa=0.3, fb=0.5, rad=0.05)
edge('og', 'r', 'ec', 'l', 'EC transferred', 'ok', (8.45, 6.45), fa=0.5, fb=0.75, rad=-0.05)
edge('seq', 'r', 'ec', 'l', '', 'ok', (0, 0), fa=0.5, fb=0.25, rad=0.0)
edge('ec', 'b', 'pw', 't', 'a reaction of\nthe pathway', 'ok', (10.85, 4.95), fa=0.5, fb=0.5)
edge('rhea', 'r', 'ec', 'b', 'join by EC\nfills the Rhea id', 'ok', (8.45, 4.55), fa=0.5, fb=0.12, rad=-0.2)
# metabolite side
edge('f', 'r', 'hyp', 'l', 'mass, adducts,\nisotope', 'weak', (3.65, 1.75), fa=0.5, fb=0.5)
edge('hyp', 'r', 'am', 'l', 'structure similarity\n(atom pairs)', 'ok', (8.45, 1.95), fa=0.5, fb=0.5)
edge('am', 't', 'pw', 'b', 'members of the\npathway', 'ok', (10.85, 2.5), fa=0.5, fb=0.5)
edge('hyp', 't', 'rhea', 'b', 'mass shifts: Rhea reactions on other substrates (analogy)', 'weak', (6.0, 2.55), fa=0.5, fb=0.5)
# failed direct tiers
fail(1.75, 4.62, 'locus, UniProt entry, own EC:\nthe pathway does not list the gene'.replace('\n', ' '))
fail(4.3, 0.45, 'exact InChIKey and first block: not in the KG')
fail(1.75, 4.15, 'OrthoDB: the other gene has no group')
ax.legend(handles=[Line2D([0], [0], color=GREEN, lw=2, label='tier that reaches the pathway'), Line2D([0], [0], color=RED, lw=0, marker='$✗$', markersize=9, label='direct tier that fails'),
                   Line2D([0], [0], color=MUTED, lw=1.7, ls=(0, (1, 2)), label='analogy only'), Line2D([0], [0], color=BLUE, lw=3.2, label='MEANtools correlation')],
          loc='lower center', ncol=4, frameon=False, fontsize=8.2, bbox_to_anchor=(0.5, -0.03))
fig.suptitle('From one MEANtools association to a PlantMetWiki pathway: which links exist', fontsize=11.5, y=0.99)
save(fig, 'tht_schema'); plt.show()

## 1. What MEANtools says

`Jeon_fungal_correlations` lists, for each gene, the metabolite features it correlates with (Pearson r, p-value). MEANtools also gives the **mutual rank (MR)**, the geometric mean of the rank of the gene among all genes correlated with the feature and of the feature among all features correlated with the gene (**1 is the best**), and groups genes and features in **functional clusters** with a cluster p-value.

In [3]:
corr = pd.read_sql('select * from Jeon_fungal_correlations', con)
corr['locus'] = corr.gene.str.extract(r'(Solyc\d+g\d+)')[0]
corr = corr.merge(pd.read_sql('select metabolite, gene, MR from Jeon_fungal_correlations_MR_edges', con), on=['metabolite', 'gene'], how='left')
feat = pd.read_csv(PILOT / 'data' / 'xcms_feature_table.csv', index_col=0)[['mzmed', 'rtmed']]
print(f"{len(corr):,} gene-feature correlations kept by MEANtools; {corr.gene.nunique():,} genes, {corr.metabolite.nunique()} features")
tht = corr[corr.locus.isin(GENES)].join(feat, on='metabolite')
(tht.groupby('metabolite').agg(genes=('locus', lambda s: ' '.join(sorted(set(s)))), r=('correlation', 'max'), P=('P', 'min'), MR=('MR', 'min'), mz=('mzmed', 'first'), rt=('rtmed', 'first'))
    .sort_values('P').head(8).round({'r': 3, 'mz': 4, 'rt': 0}))

121,043 gene-feature correlations kept by MEANtools; 5,380 genes, 138 features


,genes,r,P,MR,mz,rt
metabolite,,,,,,
M316T772,Solyc08g068710 Solyc08g068730,0.910,1.615126e-07,4.242641,316.1208,772.0
M962T828,Solyc08g068710 Solyc08g068730,0.904,2.758110e-07,9.055385,962.3389,828.0
M300T887,Solyc08g068710 Solyc08g068730,0.881,1.425104e-06,7.615773,300.1225,887.0
M301T887,Solyc08g068710 Solyc08g068730,0.878,1.658834e-06,11.045361,301.1259,887.0
M190T888,Solyc08g068710 Solyc08g068730,0.877,1.734913e-06,12.124356,190.0510,888.0
M681T918,Solyc08g068710 Solyc08g068730,0.872,2.407132e-06,6.000000,681.2360,918.0
M322T887,Solyc08g068710 Solyc08g068730,0.869,2.828186e-06,16.522712,322.1028,887.0
M316T672,Solyc08g068710 Solyc08g068730,0.824,2.610494e-05,24.819347,316.1189,672.0


Both genes correlate with the same group of features (r 0.8 to 0.9), among them `M300T887`. **Is that special?** A high correlation is common when a treatment switches many genes and metabolites on together. We rank the THT genes among all the genes that correlate with the feature.

In [4]:
f = 'M300T887'
sub = corr[corr.metabolite == f].sort_values('correlation', ascending=False).reset_index(drop=True); sub['rank'] = sub.index + 1
print(f"{f}: {len(sub)} genes are correlated with it (of {corr.gene.nunique():,}).")
print('genes with r >= 0.85:', (sub.correlation >= 0.85).sum(), '| rank of the THT genes:', sub[sub.locus.isin(GENES)][['locus', 'rank', 'correlation', 'MR']].round(3).values.tolist())
g = corr[corr.locus == 'Solyc08g068710'].sort_values('P').reset_index(drop=True)
print('seen from the gene Solyc08g068710: its features by p-value ->', g.metabolite.tolist()[:8], '| M300T887 is rank', g.index[g.metabolite == f][0] + 1, 'of', len(g))
sub.head(6)[['locus', 'rank', 'correlation', 'MR']].round(3)

M300T887: 1018 genes are correlated with it (of 5,380).
genes with r >= 0.85: 107 | rank of the THT genes: [['Solyc08g068710', 58, 0.881, 7.616], ['Solyc08g068730', 67, 0.876, 16.371]]
seen from the gene Solyc08g068710: its features by p-value -> ['M300T887', 'M301T887', 'M962T828', 'M190T888', 'M316T772', 'M322T887', 'M316T672', 'M317T672'] | M300T887 is rank 1 of 47


,locus,rank,correlation,MR
0,Solyc09g009260,1,0.949,1.000
1,Solyc02g078380,2,0.949,1.414
2,Solyc08g079420,3,0.944,1.732
3,Solyc04g064880,4,0.938,2.828
4,Solyc04g079660,5,0.936,3.873
5,Solyc04g025520,6,0.935,8.124


In [5]:
rows = []
for dr in (5, 10, 25, 50):
    q = pd.read_sql(f'select Cluster, Size, "P-value" as cluster_P, Members from Jeon_fungal_correlations_clone_DR_{dr}', con)
    for _, r in q.iterrows():
        hit = [g for g in GENES if g in r.Members]
        if hit: rows.append(dict(decay_rate=dr, cluster=r.Cluster, size=r.Size, cluster_P=round(r.cluster_P, 3), THT_in_cluster=' '.join(hit), contains_M300T887='M300T887' in r.Members))
clu = pd.DataFrame(rows); clu

,decay_rate,cluster,size,cluster_P,THT_in_cluster,contains_M300T887
0,5,150,4,0.970,Solyc08g068730,False
1,5,280,9,0.991,Solyc08g068710,True
2,10,85,20,0.975,Solyc08g068710,True
3,10,104,8,0.993,Solyc08g068730,False
4,50,190,18,0.618,Solyc08g068730,False
5,50,287,34,1.000,Solyc08g068730,False


In [6]:
fig, (a, b) = plt.subplots(1, 2, figsize=(11.5, 3.9), gridspec_kw={'width_ratios': [1.5, 1]})
a.hist(sub.correlation, bins=np.arange(0.3, 1.0001, 0.02), color=LIGHT, ec='white')
a.axvspan(0.85, 1.0, color=BLUE, alpha=0.10)
for g, c in zip(GENES, (BLUE, ORANGE)):
    r = sub[sub.locus == g].iloc[0]; a.axvline(r.correlation, color=c, lw=2); a.text(r.correlation - 0.004, a.get_ylim()[1] * (0.92 if c == BLUE else 0.78), f'{g}\nrank {int(r["rank"])}', ha='right', va='top', fontsize=8, color=c)
a.text(0.925, a.get_ylim()[1] * 0.55, f'{int((sub.correlation >= 0.85).sum())} genes\nr ≥ 0.85', ha='center', color=BLUE, fontsize=9)
a.set_xlabel('Pearson r with feature M300T887'); a.set_ylabel('number of genes'); a.set_title(f'a  The THT genes are two of {len(sub):,} genes that correlate with the feature', loc='left', fontsize=9.5, fontweight='bold')
cl = clu.sort_values('cluster_P').reset_index(drop=True)
b.barh(range(len(cl)), cl.cluster_P, color=GREY, height=0.55)
b.axvline(0.05, color=RED, lw=1.6, ls=(0, (4, 3))); b.text(0.06, len(cl) - 0.4, 'p = 0.05', color=RED, fontsize=8)
b.set_yticks(range(len(cl))); b.set_yticklabels([f'DR {d}, cluster {c}' for d, c in zip(cl.decay_rate, cl.cluster)], fontsize=8)
b.set_xlim(0, 1.05); b.invert_yaxis(); b.set_xlabel('cluster p-value (MEANtools)'); b.set_title('b  Every cluster with a THT gene is non-significant', loc='left', fontsize=9.5, fontweight='bold')
fig.tight_layout(); save(fig, 'tht_correlation_context'); plt.show()

**Reading 1.**
- The THT genes are **not the top genes** for `M300T887`: they rank about 60th and 70th of 1,018 correlated genes, and 107 genes reach r ≥ 0.85. A correlation of 0.88 therefore **does not single out the gene**; the fungus changes the expression of thousands of genes and the abundance of many features at once. From the side of the gene the picture is better: `M300T887` is the **top feature of `Solyc08g068710`** (rank 1 of its 47 features, MR 7.6). Mutual rank combines both views, which is why a pair can look strong from one side and ordinary from the other.
- The genes are in six small functional clusters at three decay rates, **none significant** (cluster p-values 0.6 to 1.0), and the two clusters that contain both the gene and the feature have p ≥ 0.97. The cluster-level statistic would have discarded this pair. It measures whether a *group* is dense, not whether a gene makes the compound.
- So the correlation is a **lead that has to be explained**, and the question of this notebook is whether the knowledge graph can explain it.

## 2. Which compound is the feature?

MEANtools annotates features by mass against a natural-products database; for this feature the match is a pterin, by mass only. We compute the exact masses of the hydroxycinnamic acid tyramine amides (HCAA) and match the XCMS features for the common adducts (10 ppm). The structures (SMILES) are written by hand and are a **hypothesis**, not an identification.

In [7]:
from rdkit import Chem, RDLogger
from rdkit.Chem.Descriptors import ExactMolWt
from rdkit.Chem.rdMolDescriptors import CalcMolFormula
RDLogger.DisableLog('rdApp.*')
HCAA = {
 'cinnamoyltyramine':  'O=C(/C=C/c1ccccc1)NCCc1ccc(O)cc1',
 '4-coumaroyltyramine':'O=C(/C=C/c1ccc(O)cc1)NCCc1ccc(O)cc1',
 'caffeoyltyramine':   'O=C(/C=C/c1ccc(O)c(O)c1)NCCc1ccc(O)cc1',
 'feruloyltyramine':   'COc1cc(/C=C/C(=O)NCCc2ccc(O)cc2)ccc1O',
 'sinapoyltyramine':   'COc1cc(/C=C/C(=O)NCCc2ccc(O)cc2)cc(OC)c1O'}
ADDUCT = {'[M+H]+': 1.007276, '[M+Na]+': 22.989218, '[M+H-H2O]+': -17.002740}
rows = []
for name, smi in HCAA.items():
    m = Chem.MolFromSmiles(smi); mass = ExactMolWt(m)
    for ad, d in ADDUCT.items():
        mz = mass + d
        for ft, r in feat[(feat.mzmed - mz).abs() / mz < 10e-6].iterrows():
            s = corr[(corr.metabolite == ft) & corr.locus.isin(GENES)]
            rows.append(dict(compound=name, formula=CalcMolFormula(m), adduct=ad, feature=ft, mz=round(r.mzmed, 4), ppm=round((r.mzmed - mz) / mz * 1e6, 1), rt=round(r.rtmed),
                             r_with_THT=round(s.correlation.max(), 2) if len(s) else None))
hits = pd.DataFrame(rows); hits[hits.compound.isin(['4-coumaroyltyramine', 'caffeoyltyramine'])]

,compound,formula,adduct,feature,mz,ppm,rt,r_with_THT
0,4-coumaroyltyramine,C17H17NO3,[M+H]+,M284T1010,284.1263,-6.4,1010,0.63
1,4-coumaroyltyramine,C17H17NO3,[M+H]+,M284T651,284.1277,-1.5,651,NaN
2,4-coumaroyltyramine,C17H17NO3,[M+H]+,M284T765,284.1259,-7.7,765,NaN
3,4-coumaroyltyramine,C17H17NO3,[M+H]+,M284T892,284.1273,-2.7,892,NaN
4,4-coumaroyltyramine,C17H17NO3,[M+Na]+,M306T1012,306.1083,-5.7,1012,0.60
5,4-coumaroyltyramine,C17H17NO3,[M+Na]+,M306T892,306.1094,-2.2,892,NaN
6,caffeoyltyramine,C17H17NO4,[M+H]+,M300T668,300.1223,-2.5,668,0.54
7,caffeoyltyramine,C17H17NO4,[M+H]+,M300T767,300.1229,-0.3,767,0.60
8,caffeoyltyramine,C17H17NO4,[M+H]+,M300T887,300.1225,-1.9,887,0.88
9,caffeoyltyramine,C17H17NO4,[M+Na]+,M322T631,322.1047,-0.7,631,NaN


In [8]:
print('features of the other two amides that correlate with the THT genes:', hits[hits.compound.isin(['feruloyltyramine', 'sinapoyltyramine'])].r_with_THT.notna().sum(), 'of', hits.compound.isin(['feruloyltyramine', 'sinapoyltyramine']).sum())
mm = pd.read_csv(PILOT / 'results' / 'metabolite_pathway_map.csv')
print('\nthe pilot mass-only mapping of the features of interest:')
print(mm[mm.feature.isin(['M284T1010', 'M300T887', 'M301T887', 'M322T887'])][['feature', 'mz', 'adduct', 'ppm_error', 'pmw_names']].drop_duplicates().to_string(index=False))
print('\nisotope check: M301T887 - M300T887 =', round(feat.mzmed['M301T887'] - feat.mzmed['M300T887'], 4), '(13C-12C = 1.0034); retention times', round(feat.rtmed['M300T887']), round(feat.rtmed['M301T887']))

features of the other two amides that correlate with the THT genes: 0 of 10

the pilot mass-only mapping of the features of interest:
  feature        mz     adduct  ppm_error             pmw_names
M284T1010 284.12630     [M+H]+      -6.40            morphinone
M284T1010 284.12630     [M+H]+      -6.40 (4-coumaroyl)tyramine
 M301T887 301.12585     [M+H]+      -7.73           salidroside
 M301T887 301.12585   [M+NH4]+       1.19             guanosine
M284T1010 284.12630 [M+H-H2O]+      -6.40          hemanthamine

isotope check: M301T887 - M300T887 = 1.0034 (13C-12C = 1.0034); retention times 887 887


In [9]:
import re as _re
smp = [c for c in feat.columns if False]  # placeholder, columns come from the full table
full = pd.read_csv(PILOT / 'data' / 'xcms_feature_table.csv', index_col=0)
cols = [c for c in full.columns if c not in ('mzmed', 'rtmed')]
meta = []
for c in cols:
    t = c.split('_'); meta.append((c, t[1], t[0]))
M = pd.DataFrame(meta, columns=['col', 'treatment', 'time'])
TIME = {'2hpi': 0, '12hpi': 1, '1dpi': 2, '2dpi': 3}; TRT = ['Mock', 'Cf', 'Chitin', 'Flg22', 'Pb', 'Sc', 'Mz', 'XeWT']
M['t'] = M.time.map(TIME); M = M.dropna(subset=['t']); M = M[M.treatment.isin(TRT)]
cond = M.groupby(['treatment', 'time']).col.apply(list).reset_index(); cond['o'] = cond.treatment.map(TRT.index) * 10 + cond.time.map(TIME); cond = cond.sort_values('o').reset_index(drop=True)
FEATS = [('M284T1010', '4-coumaroyltyramine [M+H]+'), ('M300T887', 'caffeoyltyramine [M+H]+'), ('M322T887', 'caffeoyltyramine [M+Na]+'), ('M301T887', '13C isotope of M300T887'), ('M300T767', 'same mass, other RT (767 s)')]
X = np.array([[full.loc[f, cs].astype(float).mean() for cs in cond.col] for f, _ in FEATS]); Xn = X / X.max(axis=1, keepdims=True)
rr = {f: np.corrcoef(full.loc['M300T887', [c for cs in cond.col for c in cs]].astype(float), full.loc[f, [c for cs in cond.col for c in cs]].astype(float))[0, 1] for f, _ in FEATS}
fig, ax = plt.subplots(figsize=(12.5, 3.3))
im = ax.imshow(Xn, aspect='auto', cmap='Blues', vmin=0, vmax=1)
ax.set_yticks(range(len(FEATS))); ax.set_yticklabels([f'{f}  {l}   (r {rr[f]:.2f} with M300T887)' for f, l in FEATS], fontsize=8.5)
ax.set_xticks(range(len(cond))); ax.set_xticklabels(cond.time, rotation=90, fontsize=7.5)
for i, t in enumerate(TRT):
    idx = cond.index[cond.treatment == t]
    if len(idx):
        ax.text(float(np.mean(idx)), -1.1, {'Cf': 'Cf (C. fulvum)'}.get(t, t), ha='center', fontsize=8.5, fontweight='bold' if t == 'Cf' else 'normal', color=BLUE if t == 'Cf' else INK)
        if i: ax.axvline(idx.min() - 0.5, color='white', lw=3)
for sp in ax.spines.values(): sp.set_visible(False)
cb = fig.colorbar(im, ax=ax, fraction=0.012, pad=0.01); cb.set_label('mean abundance (scaled per feature)', fontsize=8)
ax.set_title('Abundance of the features across treatments and time points (mean of the replicates)', loc='left', fontsize=10, pad=40)
fig.tight_layout(); save(fig, 'tht_feature_profiles'); plt.show()
print({f: round(v, 2) for f, v in rr.items()})

{'M284T1010': np.float64(0.81), 'M300T887': np.float64(1.0), 'M322T887': np.float64(0.98), 'M301T887': np.float64(1.0), 'M300T767': np.float64(0.9)}


**Reading 2.**
- `M300T887` fits caffeoyltyramine as [M+H]+ (-1.9 ppm), and `M322T887` fits its [M+Na]+ at the **same retention time (887 s)**, with the same correlation with the THT genes (0.88 and 0.87). Two adducts at one retention time is stronger than one mass. But the mass of caffeoyltyramine is shared by **three** features at different retention times (668, 767, 887 s): they are isomers or other compounds, and only the T887 one correlates strongly.
- `M301T887` is the **13C isotope** of `M300T887` (+1.0034, same retention time). The pilot's mass-only mapping gave it a **false identity** (salidroside and guanosine). Isotopes and adducts also **inflate the number of associations**: three features, one compound.
- `M284T1010` fits 4-coumaroyltyramine (in PlantMetWiki) but the same mass maps to morphinone and hemanthamine: **mass alone is ambiguous**. Its correlation with the THT genes is weaker (0.63).
- The features of feruloyl- and sinapoyltyramine do **not** correlate with these genes (empty r), so the picture is not "all amides follow the THT".
- The **abundance profiles** (figure above) show it: the [M+H]+ and [M+Na]+ adducts and the 13C isotope co-vary almost perfectly across the 24 conditions (r 0.98-1.00), as one compound should, and all respond most strongly to *C. fulvum* at 12 hpi. The feature of the same mass at 767 s shares that early peak (r 0.90 overall) but differs at 2 dpi and after chitin, so it is another compound or an isomer.
- This is **identification level 3-4** (mass, retention time and a plausible biochemical family). The MS/MS spectrum or a standard would confirm it.

## 3. Gene side: at which tier does the gene reach the pathway?

The pathway is looked up **live** in PlantMetWiki by its PlantCyc id, because the PlantMetWiki numbers change between releases (the pilot snapshot of 2026-09-29 calls this pathway PC712; the release r17 used here calls it PC536). Joining on the PlantCyc id is safe, joining on the PC number is not.

In [10]:
PMW = 'https://plantmetwiki.bioinformatics.nl/sparql'
def sparql(q, ep=PMW):
    r = requests.get(ep, params={'query': q}, headers={'Accept': 'application/sparql-results+json'}, timeout=120)
    return [{k: v['value'] for k, v in b.items()} for b in r.json()['results']['bindings']]
pw = sparql(f'SELECT ?pw WHERE {{ ?pw <http://xmlns.com/foaf/0.1/page> <https://pmn.plantcyc.org/pathway?orgid=PLANT&id={PLANTCYC}> }}')
PW_IRI = pw[0]['pw']; PC = re.search(r'/(PC\d+)_', PW_IRI).group(1)
print(PLANTCYC, '->', PW_IRI.split('/')[-1], '(in the release r17); the pilot snapshot has it as PC712')
KV = '''PREFIX v: <http://rdf-plantmetwiki.bioinformatics.nl/vocab/>
PREFIX gpml: <http://vocabularies.wikipathways.org/gpml#>
PREFIX dcterms: <http://purl.org/dc/terms/>
SELECT ?e ?key ?val WHERE { ?e a gpml:%s ; dcterms:isPartOf ?gp ; v:gpmlProperty ?b . FILTER(REGEX(STR(?gp), "/Pathway/%s_")) ?b v:key ?key ; v:value ?val . FILTER(?key IN (%s)) }'''
rx = pd.DataFrame(sparql(KV % ('Interaction', PC, '"EC_Number","AlternativeId_Rhea","AlternativeId_Uniprot","ReactionEquation"')))
rx['e'] = rx.e.str.split('/').str[-1]
rxt = rx.pivot_table(index='e', columns='key', values='val', aggfunc=lambda s: '; '.join(sorted(set(s))))
print('Reactions of the pathway in PlantMetWiki (EC, Rhea id and UniProt enzyme where given):'); rxt

PWY-5474 -> PC536_r17.0.0_20260605-171052 (in the release r17); the pilot snapshot has it as PC712


Reactions of the pathway in PlantMetWiki (EC, Rhea id and UniProt enzyme where given):


key,AlternativeId_Rhea,AlternativeId_Uniprot,EC_Number,ReactionEquation
e,,,,
RXN_8713,NaN,NaN,EC-2.3.1.110,TYRAMINE + CINNAMOYL-COA => PROTON + CPD-8941 + CO-A
RXN_8714,NaN,NaN,EC-2.3.1.110,TYRAMINE + SINAPOYL-COA => PROTON + CPD-8942 + CO-A
RXN_8715,NaN,NaN,EC-2.3.1.110,P-COUMAROYL-COA + TYRAMINE => PROTON + CPD-8943 + CO-A
TYRAMINE_N_FERULOYLTRANSFERASE_RXN,NaN,NaN,EC-2.3.1.110,FERULOYL-COA + TYRAMINE <=> PROTON + CPD-440 + CO-A
TYROSINE_DECARBOXYLASE_RXN,14345,P54771,NaN,PROTON + TYR => CARBON-DIOXIDE + TYRAMINE


In [11]:
dn = pd.DataFrame(sparql(KV % ('DataNode', PC, '"AlternativeId_Uniprot","AlternativeId_Sgn","Synonym_1","Synonym_2"')))
pc_ec = {m for v in rx[rx.key == 'EC_Number'].val for m in re.findall(r'\d+\.\d+\.\d+\.[\d-]+', v)}
pc_up = set(rx[rx.key == 'AlternativeId_Uniprot'].val) | set(dn[dn.key == 'AlternativeId_Uniprot'].val)
pc_sl = {m for v in dn.val for m in re.findall(r'Solyc\d+g\d+', v)}
print('EC numbers of the pathway :', sorted(pc_ec)); print('UniProt enzymes           :', sorted(pc_up)); print('Solyc loci in the pathway :', sorted(pc_sl) or 'none')

EC numbers of the pathway : ['2.3.1.110']
UniProt enzymes           : ['P54771', 'Q1KLR8', 'Q9M0G4']
Solyc loci in the pathway : none


In [12]:
up = pd.read_csv(PILOT / 'data' / 'uniprot_tomato.tsv.gz', sep='\t').set_index('Entry')
rows = []
for locus, acc in GENES.items():
    ec_own = up['EC number'].get(acc) if acc in up.index else None
    rows.append({'gene': locus, 'uniprot': acc, 'protein': up['Protein names'].get(acc), 'tier gene_locus': locus in pc_sl, 'tier gene_uniprot': acc in pc_up, 'own EC': ec_own if isinstance(ec_own, str) else None,
                 'tier gene_ec': bool(isinstance(ec_own, str) and set(re.findall(r'[\d.]+', ec_own)) & pc_ec)})
pd.DataFrame(rows)

,gene,uniprot,protein,tier gene_locus,tier gene_uniprot,own EC,tier gene_ec
0,Solyc08g068730,A0A3Q7HPP9,N-hydroxycinnamoyl-CoA:tyramine N-hydroxycinnamoyl transferase THT1-3,False,False,None,False
1,Solyc08g068710,A0A3Q7HQ14,N-acetyltransferase domain-containing protein,False,False,None,False


**The three direct tiers fail, for a clear reason on both sides.** PlantMetWiki lists the THT step **by EC number only (2.3.1.110), without a UniProt enzyme**; the only UniProt enzymes of the pathway are the tyrosine decarboxylases that make tyramine, an upstream step. The tomato THT proteins are unreviewed entries **without an EC number**. The information exists, but the annotation is missing on both sides.

**Orthology (OrthoDB, as in the tier-5 query).** We ask OrthoDB for the orthologous groups of each tomato protein and list the group members that carry EC 2.3.1.110 in UniProt. The Arabidopsis enzyme of the pathway is the control.

In [13]:
ODB = 'https://sparql.orthodb.org/sparql/'
def odb(q):
    r = requests.get(ODB, params={'query': q}, headers={'Accept': 'application/sparql-results+json'}, timeout=120); return r.json()['results']['bindings']
PFX = 'PREFIX odb: <http://purl.orthodb.org/>\nPREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>\n'
def groups(acc): return [b['og']['value'].split('/')[-1] for b in odb(PFX + f'SELECT ?og WHERE {{ ?og a odb:OrthoGroup ; odb:hasMember/rdfs:seeAlso <http://purl.uniprot.org/uniprot/{acc}> }}')]
def members(og):
    b = odb(PFX + f'SELECT ?u WHERE {{ <http://purl.orthodb.org/odbgroup/{og}> odb:hasMember/rdfs:seeAlso ?u . FILTER(STRSTARTS(STR(?u), "http://purl.uniprot.org/uniprot/")) }} LIMIT 400')
    return [x['u']['value'].split('/')[-1] for x in b]
def with_ec(accs, ec='2.3.1.110'):
    if not accs: return pd.DataFrame()
    q = '(' + ' OR '.join(f'accession:{a}' for a in accs[:200]) + f') AND (ec:{ec})'
    r = requests.get('https://rest.uniprot.org/uniprotkb/search', params={'query': q, 'fields': 'accession,organism_name,reviewed,protein_name', 'format': 'tsv'}, timeout=60)
    return pd.read_csv(io.StringIO(r.text), sep='\t')
ortho = []
for acc in list(GENES.values()) + ['Q9M0G4']:
    gs = groups(acc); print(acc, '->', gs or 'no orthologous group in OrthoDB')
    for og in gs:
        e = with_ec(members(og))
        if len(e): ortho.append((acc, og, e))
for acc, og, e in ortho[:1]: print(f'\nmembers of {og} (group of {acc}) with EC 2.3.1.110:'); print(e.to_string(index=False))

A0A3Q7HPP9 -> no orthologous group in OrthoDB
A0A3Q7HQ14 -> ['42360at4069', '1672580at2759', '806928at3193', '884189at33090', '482473at71240']


Q9M0G4 -> ['1295357at71240', '1599196at3193', '2566518at2759', '1778127at33090', '51539at3699']



members of 42360at4069 (group of A0A3Q7HQ14) with EC 2.3.1.110:
 Entry                           Organism   Reviewed                                                                                                      Protein names
P80969 Nicotiana tabacum (Common tobacco)   reviewed Tyramine N-feruloyltransferase 10/30 (EC 2.3.1.110) (Hydroxycinnamoyl-CoA: tyramine N-hydroxycinnamoyltransferase)
Q9SMB8 Nicotiana tabacum (Common tobacco)   reviewed  Tyramine N-feruloyltransferase 4/11 (EC 2.3.1.110) (Hydroxycinnamoyl-CoA: tyramine N-hydroxycinnamoyltransferase)
W8E5Y4         Solanum tuberosum (Potato) unreviewed                                                               Tyramine hydroxycinnamoyl transferase (EC 2.3.1.110)


**Orthology works for one gene and not for the other.** `Solyc08g068710` is in an OrthoDB group (built at the genus *Solanum* level) with the potato and tobacco **THT enzymes that carry EC 2.3.1.110**, so the pathway is reached by orthology plus a transferred EC. `Solyc08g068730` is **not in OrthoDB at all** (the Arabidopsis control is), although it is the better-annotated entry. A tier that depends on a database covering the identifier has to be reported with that caveat: the failure is about the identifier, not about the biology.

**Sequence similarity to EC 2.3.1.110 enzymes** replaces the database for the gene without a group: global alignment of each tomato protein with the plant proteins of UniProt annotated with EC 2.3.1.110.

In [14]:
from Bio import Align
def fasta(acc): return requests.get(f'https://rest.uniprot.org/uniprotkb/{acc}.fasta', timeout=30).text.split('\n', 1)[1].replace('\n', '')
ref = pd.read_csv(io.StringIO(requests.get('https://rest.uniprot.org/uniprotkb/search', params={'query': '(ec:2.3.1.110) AND (taxonomy_id:33090)', 'fields': 'accession,organism_name,reviewed,protein_name,sequence', 'format': 'tsv', 'size': 200}, timeout=60).text), sep='\t')
ref.columns = ['acc', 'organism', 'status', 'protein', 'seq']
al = Align.PairwiseAligner(); al.substitution_matrix = Align.substitution_matrices.load('BLOSUM62'); al.open_gap_score = -10; al.extend_gap_score = -0.5; al.mode = 'global'
def identity(a, b):
    aln = al.align(a, b)[0]
    return sum(a[i] == b[j] for (s1, e1), (s2, e2) in zip(*aln.aligned) for i, j in zip(range(s1, e1), range(s2, e2))) / max(len(a), len(b))
res = [dict(gene=l, EC_enzyme=r.acc, organism=r.organism, status=r.status, identity=round(identity(fasta(a), r.seq), 2)) for l, a in GENES.items() for _, r in ref.iterrows()]
seqtab = pd.DataFrame(res); seqtab.sort_values('identity', ascending=False).groupby('gene').head(4).to_string(index=False)

'          gene EC_enzyme                           organism     status  identity\nSolyc08g068710    W8E5Y4         Solanum tuberosum (Potato) unreviewed      0.89\nSolyc08g068710    W8E402         Solanum tuberosum (Potato) unreviewed      0.87\nSolyc08g068730    W8E5Y4         Solanum tuberosum (Potato) unreviewed      0.86\nSolyc08g068730    W8E402         Solanum tuberosum (Potato) unreviewed      0.83\nSolyc08g068730    Q9SMB8 Nicotiana tabacum (Common tobacco)   reviewed      0.76\nSolyc08g068730    P80969 Nicotiana tabacum (Common tobacco)   reviewed      0.76\nSolyc08g068710    Q9SMB8 Nicotiana tabacum (Common tobacco)   reviewed      0.73\nSolyc08g068710    P80969 Nicotiana tabacum (Common tobacco)   reviewed      0.73'

Both tomato proteins are **73-89% identical to THT enzymes with EC 2.3.1.110** (the tobacco ones reviewed), far above the ~40% usually needed to transfer an EC number, and the pathway lists EC 2.3.1.110. The gene without an OrthoDB group is reached by **sequence similarity**, and `Solyc08g068710`, annotated only as an "N-acetyltransferase domain-containing protein", is shown to be a THT too, which a keyword search would not find. Caution: the transfer rests on an EC number that a few unrelated acyltransferases also carry (identities of 0.1-0.4 in the full table), so the threshold matters.

**Co-expression with other genes of the pathway.** If the THT acts in a coordinated response, the enzyme that makes tyramine should follow the same compound. The pathway lists the **tyrosine decarboxylase** (UniProt `P54771`, `Q9M0G4`, `Q1KLR8`). The tomato genes annotated as aromatic amino acid decarboxylases have no EC number in UniProt, so we try to link them to these enzymes by sequence, as before, and then read their correlation with the feature.

In [15]:
aadc = up[up['Protein names'].str.contains('decarboxylase', case=False, na=False) & up['Protein names'].str.contains('aromatic', case=False, na=False)].copy()
aadc['locus'] = aadc.EnsemblPlants.astype(str).str.extract(r'(Solyc\d+g\d+)')[0]; aadc = aadc.dropna(subset=['locus'])
pw_enz = {'P54771': 'Papaver TyDC', 'Q9M0G4': 'Arabidopsis TyDC2'}
rows = []
for acc, r in aadc.iterrows():
    ids = {n: round(identity(fasta(acc), fasta(a)), 2) for a, n in pw_enz.items()}
    c = corr[(corr.locus == r.locus) & corr.metabolite.isin(['M300T887', 'M322T887'])]
    rows.append({'locus': r.locus, 'uniprot': acc, 'protein': r['Protein names'], **{f'identity to {n}': v for n, v in ids.items()}, 'linked by sequence (>= 0.40)': max(ids.values()) >= 0.40, 'r with M300T887': c[c.metabolite == 'M300T887'].correlation.max()})
# also the neighbour gene Solyc08g068690, which is not annotated as a decarboxylase
print('tomato genes annotated as aromatic amino acid decarboxylases: do they link to the decarboxylases of the pathway by sequence, and do they follow the feature?')
aadc_tab = pd.DataFrame(rows).round(3); aadc_tab

tomato genes annotated as aromatic amino acid decarboxylases: do they link to the decarboxylases of the pathway by sequence, and do they follow the feature?


,locus,uniprot,protein,identity to Papaver TyDC,identity to Arabidopsis TyDC2,linked by sequence (>= 0.40),r with M300T887
0,Solyc08g006740,A0A3Q7HJX8,Aromatic amino acid decarboxylase 2,0.20,0.21,False,0.889
1,Solyc08g068680,A0A3Q7IL88,Aromatic amino acid decarboxylase 1A,0.22,0.22,False,0.598


In [16]:
bars = []
for g in GENES:
    for _, r in seqtab[seqtab.gene == g].sort_values('identity', ascending=False).head(3).iterrows():
        bars.append((f"{g}  vs  {r.EC_enzyme} ({r.organism.split(' (')[0]})", r.identity, BLUE))
for _, r in aadc_tab.iterrows():
    for col, nm in (('identity to Papaver TyDC', 'Papaver TyDC'), ('identity to Arabidopsis TyDC2', 'Arabidopsis TyDC2')):
        bars.append((f"{r.locus} (AADC)  vs  {nm}", r[col], ORANGE))
fig, ax = plt.subplots(figsize=(9.5, 4.8))
y = np.arange(len(bars))[::-1]
ax.barh(y, [b[1] for b in bars], color=[b[2] for b in bars], height=0.62)
for yy, b in zip(y, bars): ax.text(b[1] + 0.01, yy, f'{b[1]:.2f}', va='center', fontsize=8)
ax.axvline(0.40, color=RED, lw=1.6, ls=(0, (4, 3))); ax.text(0.42, 2.5, 'about 40 %: enough to transfer an EC number', color=RED, fontsize=8, va='center')
ax.set_yticks(y); ax.set_yticklabels([b[0] for b in bars], fontsize=8); ax.set_xlim(0, 1.0); ax.set_xlabel('sequence identity (global alignment)')
ax.legend(handles=[Patch(fc=BLUE, label='THT proteins vs enzymes with EC 2.3.1.110'), Patch(fc=ORANGE, label='decarboxylases vs the tyrosine decarboxylases of the pathway')], loc='lower right', frameon=False, fontsize=8)
ax.set_title('Sequence similarity links the THT genes to the pathway, and fails for the decarboxylases', loc='left', fontsize=10, fontweight='bold')
fig.tight_layout(); save(fig, 'tht_sequence_identity'); plt.show()

In [17]:
print('other genes of the same chromosome-8 neighbourhood (Solyc08g0686xx-0687xx) in the MEANtools table, with their correlation with M300T887:')
nb = corr[corr.locus.str.match(r'Solyc08g06(8[6-9]|87)\d\d', na=False) & (corr.metabolite == 'M300T887')][['locus', 'correlation', 'P', 'MR']].round(3).sort_values('P'); nb

other genes of the same chromosome-8 neighbourhood (Solyc08g0686xx-0687xx) in the MEANtools table, with their correlation with M300T887:


,locus,correlation,P,MR
38636,Solyc08g068710,0.881,0.000,7.616
38645,Solyc08g068730,0.876,0.000,16.371
38647,Solyc08g068790,0.874,0.000,8.307
38727,Solyc08g068780,0.816,0.000,57.254
38893,Solyc08g068690,0.686,0.002,25.100
38957,Solyc08g068610,0.645,0.004,27.532
38981,Solyc08g068770,0.630,0.005,28.390
39036,Solyc08g068680,0.598,0.009,42.802
39419,Solyc08g068870,-0.567,0.014,100.459


**Reading 3.**
- **`Solyc08g006740` (an aromatic amino acid decarboxylase) correlates with `M300T887` at 0.89, as strongly as the THT.** Biologically that is what we would expect, since the decarboxylase makes the tyramine that the THT uses. But the **sequence link fails**: the two tomato decarboxylases are only about **20-25% identical** to the pathway's tyrosine decarboxylases, well below the ~40% needed to transfer a function, so by the rules of the tiers **the co-expression tier does not fire**: the gene cannot be recognised as a gene of the pathway. Recognising it takes expert knowledge (plant aromatic amino acid decarboxylases act on tyrosine) or a curated annotation, not a tier. This is a good example of a **true signal that a rule-based tier misses**, and of why the correlation is kept as supporting information.
- The correlation also extends to a **neighbourhood on chromosome 8** (a run of consecutive gene models, `Solyc08g068610-068790`): the THT genes and `Solyc08g068780`, `068790` all correlate with `M300T887` (r 0.82-0.88). A gene-cluster tier (a plantiSMASH cluster) would be a further, weaker line of evidence; it was not tested here.
- **Summary of the gene side:** locus, UniProt and own EC fail; orthology works for one gene; sequence similarity to EC-annotated enzymes works for both; co-expression with a second pathway gene is present but cannot be counted. Every tier that works relies on **annotation transferred from other species**.

## 4. Metabolite side: at which tier does the feature reach the pathway?

Caffeoyltyramine is **not in PlantMetWiki**. We compute its InChIKey, compare it with all PlantMetWiki metabolites (exact key, first block), then rank the PlantMetWiki metabolites by chemical similarity (Tanimoto) and see which pathways the most similar ones belong to. A similarity score depends on the fingerprint, so we try five, and report all.

In [18]:
from rdkit.Chem import inchi, DataStructs, rdFingerprintGenerator as G
q = Chem.MolFromSmiles(HCAA['caffeoyltyramine']); qkey = inchi.MolToInchiKey(q)
pmw = pd.read_csv(PILOT / 'data' / 'plantmetwiki_metabolite_inchi.csv')
names = pd.read_csv(MET / 'plantmetwiki_metabolites.csv')[['inchikey', 'name']].set_index('inchikey').name
pwm = pd.read_csv(MET / 'plantmetwiki_metabolite_pathways.csv')
pwb = pwm.assign(b=pwm.inchikey.str[:14]).groupby('b').pathway.apply(set)
print('InChIKey of the hypothesis:', qkey)
print('exact InChIKey in PlantMetWiki:', (pmw.inchikey == qkey).any(), '| first block (connectivity) in PlantMetWiki:', (pmw.inchikey.str[:14] == qkey[:14]).any(), '=> the direct tiers fail')
mols = [(k, Chem.MolFromInchi(i)) for k, i in zip(pmw.inchikey, pmw.inchi) if isinstance(i, str)]
mols = [(k, m) for k, m in mols if m is not None and m.GetNumAtoms() > 5]
PATHWAY_NAME = 'hydroxycinnamic acid tyramine amides biosynthesis'
FPS = {'Morgan r2, bits': (G.GetMorganGenerator(radius=2, fpSize=2048), False), 'Morgan r2, counts': (G.GetMorganGenerator(radius=2, fpSize=2048), True),
       'atom pairs, counts': (G.GetAtomPairGenerator(fpSize=2048), True), 'topological torsion': (G.GetTopologicalTorsionGenerator(fpSize=2048), True), 'RDKit path': (G.GetRDKitFPGenerator(fpSize=2048), False)}
tab, sims = [], {}
for n, (g, cnt) in FPS.items():
    fp = (lambda m: g.GetCountFingerprint(m)) if cnt else (lambda m: g.GetFingerprint(m))
    qf = fp(q); s = pd.Series({k: DataStructs.TanimotoSimilarity(qf, fp(m)) for k, m in mols}).sort_values(ascending=False); sims[n] = s
    inp = [i + 1 for i, k in enumerate(s.index) if PATHWAY_NAME in pwb.get(k[:14], set())]
    tab.append({'fingerprint': n, 'best hit': f'{names.get(s.index[0])} ({s.iloc[0]:.2f})', 'best compound of the pathway': f'rank {inp[0]}' if inp else '-', 'pathway compounds in top 10': sum(r <= 10 for r in inp), 'compounds compared': len(s)})
pd.DataFrame(tab)

InChIKey of the hypothesis:

 VSHUQLRHTJOKTA-XBXARRHUSA-N
exact InChIKey in PlantMetWiki: False | first block (connectivity) in PlantMetWiki: False => the direct tiers fail


,fingerprint,best hit,best compound of the pathway,pathway compounds in top 10,compounds compared
0,"Morgan r2, bits",butein (0.51),rank 7,2,3962
1,"Morgan r2, counts",None (0.63),rank 3,2,3962
2,"atom pairs, counts",(4-coumaroyl)tyramine (0.58),rank 1,3,3962
3,topological torsion,None (0.64),rank 4,1,3962
4,RDKit path,caffealdehyde (0.61),rank 12,0,3962


### Cross-check of the masses and of the structure in public databases

The masses above are **computed** from structures that were written by hand (RDKit, exact mass of the formula). To make sure that the hand-written structure of the hypothesis is not a typo and that the masses are right, the same compounds are looked up in **PubChem** (by InChIKey, so that the structure is compared and not only a name) and in **ChEBI** (through the EBI Ontology Lookup Service). This checks the **structure and the mass of the hypothesis**; it does not identify the feature (many compounds have this mass).

In [19]:
import requests
from rdkit.Chem.Descriptors import ExactMolWt
from rdkit.Chem.rdMolDescriptors import CalcMolFormula

HYP = {'caffeoyltyramine (the hypothesis)': HCAA['caffeoyltyramine'], 'caffeic acid (the implied acid)': 'OC(=O)/C=C/c1ccc(O)c(O)c1', 'tyramine': 'NCCc1ccc(O)cc1'}
def pubchem(key):
    r = requests.get(f'https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/inchikey/{key}/property/Title,MolecularFormula,ExactMass/JSON', timeout=60)
    return r.json()['PropertyTable']['Properties'][0] if r.ok else {}
def chebi(name):
    r = requests.get('https://www.ebi.ac.uk/ols4/api/search', params={'q': name, 'ontology': 'chebi', 'rows': 1, 'exact': 'false'}, timeout=60)
    d = r.json()['response']['docs'] if r.ok else []
    return (d[0]['obo_id'], d[0]['label']) if d else (None, None)
rows = []
for n, smi in HYP.items():
    m = Chem.MolFromSmiles(smi); key = inchi.MolToInchiKey(m)
    pc = pubchem(key)
    cid, lab = chebi(n.split(' (')[0])
    rows.append({'compound': n, 'formula (RDKit)': CalcMolFormula(m), 'exact mass (RDKit)': round(ExactMolWt(m), 4), 'InChIKey (from my structure)': key,
                 'PubChem title': pc.get('Title'), 'PubChem formula': pc.get('MolecularFormula'), 'PubChem exact mass': round(float(pc['ExactMass']), 4) if pc else None,
                 'same InChIKey in PubChem': bool(pc), 'ChEBI': f'{cid} {lab}' if cid else None})
xc = pd.DataFrame(rows); print('Structure and mass of the hypothesis against PubChem (looked up by InChIKey) and ChEBI:'); xc.set_index('compound').T

Structure and mass of the hypothesis against PubChem (looked up by InChIKey) and ChEBI:


compound,caffeoyltyramine (the hypothesis),caffeic acid (the implied acid),tyramine
formula (RDKit),C17H17NO4,C9H8O4,C8H11NO
exact mass (RDKit),299.1158,180.0423,137.0841
InChIKey (from my structure),VSHUQLRHTJOKTA-XBXARRHUSA-N,QAIPRVGONGVQAS-DUXPYHPUSA-N,DZGWFCGJZKJUFP-UHFFFAOYSA-N
PubChem title,N-Caffeoyltyramine,Caffeic Acid,Tyramine
PubChem formula,C17H17NO4,C9H8O4,C8H11NO
PubChem exact mass,299.1158,180.0423,137.0841
same InChIKey in PubChem,True,True,True
ChEBI,CHEBI:193479 N-cis-Caffeoyltyramine,CHEBI:36281 caffeic acid,CHEBI:15760 tyramine


In [20]:
rng = np.random.default_rng(1)
fig, axes = plt.subplots(len(sims), 1, figsize=(10.5, 6.8), sharex=True)
for ax, (n, s_) in zip(axes, sims.items()):
    inpw = np.array([PATHWAY_NAME in pwb.get(k[:14], set()) for k in s_.index])
    ax.scatter(s_.values[~inpw], rng.uniform(-0.35, 0.35, (~inpw).sum()), s=3, color=LIGHT, lw=0, rasterized=True)
    ax.scatter(s_.values[inpw], rng.uniform(-0.2, 0.2, inpw.sum()), s=34, color=BLUE, ec='white', lw=0.6, zorder=3)
    rk = [int(i) + 1 for i in np.where(inpw)[0]]
    ax.scatter([s_.values[0]], [0], s=70, facecolors='none', edgecolors=ORANGE, lw=1.6, zorder=4)
    ax.text(1.005, 0, f'best hit: {names.get(s_.index[0]) or "(unnamed)"}\npathway compounds at ranks {rk[:4]}', va='center', fontsize=7.6, transform=ax.get_yaxis_transform())
    ax.set_yticks([]); ax.set_ylabel(n, rotation=0, ha='right', va='center', fontsize=8.5); ax.spines['left'].set_visible(False)
axes[-1].set_xlabel('Tanimoto similarity to the caffeoyltyramine hypothesis (each dot is a PlantMetWiki compound)')
fig.legend(handles=[Line2D([0], [0], marker='o', color='w', markerfacecolor=BLUE, markersize=7, label='compound of the pathway hydroxycinnamic acid tyramine amides'), Line2D([0], [0], marker='o', color='w', markerfacecolor=LIGHT, markersize=5, label='other PlantMetWiki compounds'),
                    Line2D([0], [0], marker='o', color='w', markerfacecolor='none', markeredgecolor=ORANGE, markersize=9, label='highest similarity')], loc='upper center', ncol=3, frameon=False, fontsize=8, bbox_to_anchor=(0.45, 1.03))
fig.tight_layout(rect=(0, 0, 0.8, 0.97)); save(fig, 'tht_fingerprints'); plt.show()

In [21]:
s = sims['atom pairs, counts']
top = pd.DataFrame({'tanimoto': s.head(12)}).assign(name=lambda d: [names.get(k) for k in d.index], pathways=lambda d: [sorted(pwb.get(k[:14], set()))[:3] for k in d.index])
print('top 12 PlantMetWiki compounds by atom-pair similarity to the caffeoyltyramine hypothesis:'); top.round(2)

top 12 PlantMetWiki compounds by atom-pair similarity to the caffeoyltyramine hypothesis:


,tanimoto,name,pathways
RXGUTQNKCXHALN-BJMVGYQFSA-N,0.58,(4-coumaroyl)tyramine,[hydroxycinnamic acid tyramine amides biosynthesis]
NPNNKDMSXVRADT-WEVVVXLNSA-N,0.58,N-feruloyltyramine,"[hydroxycinnamic acid tyramine amides biosynthesis, suberin monome..."
ZIUGAQNCXNLSFT-KZJSRBBCSA-M,0.57,NaN,[]
KIMHJUCTTIEANQ-KZJSRBBCSA-M,0.53,NaN,[]
YXKFALZVRFVXFA-QHHAFSJGSA-N,0.51,caffeoylserotonin,[hydroxycinnamic acid serotonin amides biosynthesis]
WLZPAFGVOWCVMG-FPYGCLRLSA-N,0.49,(4-coumaroyl)serotonin,[hydroxycinnamic acid serotonin amides biosynthesis]
KGOYCHSKGXJDND-DHZHZOJOSA-N,0.48,cinnamoyltyramine,[hydroxycinnamic acid tyramine amides biosynthesis]
DOUMFZQKYFQNTF-WUTVXBCWSA-M,0.48,(R)-rosmarinate,"[rosmarinic acid biosynthesis I, rosmarinic acid biosynthesis II, ..."
LVPGCTXCBGXZHO-ZWIJEDICSA-M,0.48,NaN,[]
NDNXUPPJUCZTKB-KBXRYBNXSA-N,0.46,NaN,[]


In [22]:
# which pathways do the similar compounds lead to? (aggregate the top 30 by pathway)
agg = {}
for n, s in sims.items():
    t = s.head(30)
    for k, v in t.items():
        for p in pwb.get(k[:14], set()): agg.setdefault(n, {}).setdefault(p, []).append(v)
out = []
for n, d in agg.items():
    best = sorted(d.items(), key=lambda kv: (-len(kv[1]), -max(kv[1])))[:3]
    out.append({'fingerprint': n, 'pathways with most compounds in the top 30': '; '.join(f'{p} ({len(v)})' for p, v in best)})
pd.DataFrame(out)

,fingerprint,pathways with most compounds in the top 30
0,"Morgan r2, bits",Amaryllidacea alkaloids biosynthesis (5); aurone biosynthesis (4);...
1,"Morgan r2, counts",hydroxycinnamic acid tyramine amides biosynthesis (4); aurone bios...
2,"atom pairs, counts",hydroxycinnamic acid tyramine amides biosynthesis (4); hydroxycinn...
3,topological torsion,Amaryllidacea alkaloids biosynthesis (6); suberin monomers biosynt...
4,RDKit path,free phenylpropanoid acid biosynthesis (6); suberin monomers biosy...


**Reading 4.**
- The direct tiers fail (neither the exact InChIKey nor the first block is in PlantMetWiki), so the compound is **new to the graph**.
- **A similarity tier depends on the fingerprint.** With the usual Morgan bit fingerprint the nearest compounds are **chalcones** (butein, aurone pathways; Tanimoto 0.51) and the pathway compounds come only at rank 7 and 10: a threshold of 0.5 would send the metabolite to the **wrong pathway** and miss the right one. With **atom-pair counts** the two nearest compounds are the pathway's own amides (4-coumaroyltyramine and N-feruloyltyramine, 0.58). Aggregating the top 30 by pathway puts the pathway first for the two count-based fingerprints (Morgan counts, atom pairs) and not for the other three (bit Morgan, torsion and RDKit path favour other pathways). We looked at five fingerprints *knowing the answer*, so the choice of atom pairs is not validated; it has to be fixed **before** a calibration on pathways we do not know, together with a threshold and the rule that aggregates by pathway (several similar compounds in one pathway count more than one).
- The similarity tier gives a **pathway of related compounds, not an identity**: chalcone and caffeate pathways are in the lists as well (they share the caffeoyl group), and they are the false leads that the score has to keep below.
- The same pathway is reached from the **gene side** (sequence, orthology) and from the **metabolite side** (structure): the agreement of two independent routes is the evidence.

## 5. Reaction side: EC, Rhea and the mass-shift ladder

What does PlantMetWiki have for the THT step, what does Rhea have, and are the compound and its relatives connected by known reactions?

In [23]:
def rhea(query):
    r = requests.get('https://www.rhea-db.org/rhea', params={'query': query, 'columns': 'rhea-id,equation,ec', 'format': 'tsv'}, timeout=60)
    return pd.read_csv(io.StringIO(r.text), sep='\t') if r.text.strip() else pd.DataFrame()
tht_rhea = rhea('ec:2.3.1.110')
print('PlantMetWiki reactions of the pathway with EC 2.3.1.110 and their Rhea ids:')
print(rxt[rxt.EC_Number.fillna('').str.contains('2.3.1.110')][['EC_Number', 'AlternativeId_Rhea']].to_string())
print('\nRhea reactions with EC 2.3.1.110:'); print(tht_rhea.to_string(index=False))
print('\nRhea reactions with caffeoyltyramine or with caffeoyl-CoA and tyramine:', len(rhea('caffeoyltyramine')), '/', len(rhea('caffeoyl tyramine')))

PlantMetWiki reactions of the pathway with EC 2.3.1.110 and their Rhea ids:
key                                    EC_Number AlternativeId_Rhea
e                                                                  
RXN_8713                            EC-2.3.1.110                NaN
RXN_8714                            EC-2.3.1.110                NaN
RXN_8715                            EC-2.3.1.110                NaN
TYRAMINE_N_FERULOYLTRANSFERASE_RXN  EC-2.3.1.110                NaN

Rhea reactions with EC 2.3.1.110:
Reaction identifier                                                                  Equation    EC number
         RHEA:19685       (E)-feruloyl-CoA + tyramine = N-[(E)-feruloyl]tyramine + CoA + H(+) EC:2.3.1.110
         RHEA:86063 (E)-4-coumaroyl-CoA + tyramine = N-[(E)-4-coumaroyl]tyramine + CoA + H(+) EC:2.3.1.110
         RHEA:86431     (E)-cinnamoyl-CoA + tyramine = N-[(E)-cinnamoyl]tyramine + CoA + H(+) EC:2.3.1.110
         RHEA:86435       (E)-sinapoyl-CoA + tyramine


Rhea reactions with caffeoyltyramine or with caffeoyl-CoA and tyramine: 0 / 0


In [24]:
print('The ring modifications of the mass-shift ladder (coumaroyl -> caffeoyl is +O, caffeoyl -> feruloyl is +CH2) exist in Rhea only on other substrates:')
for ec in ('1.14.14.96', '2.1.1.68'): print(rhea(f'ec:{ec}').head(2).to_string(index=False))
d = {'coumaroyl->caffeoyl': ExactMolWt(Chem.MolFromSmiles(HCAA['caffeoyltyramine'])) - ExactMolWt(Chem.MolFromSmiles(HCAA['4-coumaroyltyramine'])),
     'caffeoyl->feruloyl': ExactMolWt(Chem.MolFromSmiles(HCAA['feruloyltyramine'])) - ExactMolWt(Chem.MolFromSmiles(HCAA['caffeoyltyramine']))}
print('\nmass shifts:', {k: round(v, 4) for k, v in d.items()})

The ring modifications of the mass-shift ladder (coumaroyl -> caffeoyl is +O, caffeoyl -> feruloyl is +CH2) exist in Rhea only on other substrates:


Reaction identifier                                                                                                                                                              Equation     EC number
         RHEA:16265 trans-5-O-(4-coumaroyl)-D-quinate + reduced [NADPH--hemoprotein reductase] + O2 = 5-O-[(E)-caffeoyl]-D-quinate + oxidized [NADPH--hemoprotein reductase] + H2O + H(+) EC:1.14.14.96


Reaction identifier                                                                                 Equation   EC number
         RHEA:20225 (E)-caffeate + S-adenosyl-L-methionine = (E)-ferulate + S-adenosyl-L-homocysteine + H(+) EC:2.1.1.68

mass shifts: {'coumaroyl->caffeoyl': 15.9949, 'caffeoyl->feruloyl': 14.0157}


**Reading 5.**
- PlantMetWiki lists **four THT reactions with EC 2.3.1.110 and no Rhea id**, while Rhea has **four reactions for EC 2.3.1.110** (feruloyl-, coumaroyl-, cinnamoyl- and sinapoyl-CoA with tyramine). The reaction of the pathway is therefore found at the **EC tier**, and the **Rhea tier can be filled by joining the EC number to Rhea**: a gap of the knowledge graph that this lookup closes.
- Rhea has **no reaction for caffeoyl-CoA with tyramine** or for caffeoyltyramine. The compound cannot be reached through Rhea by identity. It is connected to its relatives only by the **mass shifts** (+15.9949 for the hydroxylation and +14.0157 for the methylation, which exist in Rhea on other substrates). That is an analogy, the weakest reaction-level tier, and the reason the `rhea-step-between-metabolites` tier is separate and weighted lower.
- If caffeoyltyramine is real and made by the THT from caffeoyl-CoA, the reaction is **missing from PlantMetWiki and from Rhea**: a candidate for **extending the pathway**, which is the purpose of the whole prioritization.

## 6. Summary: which tiers retrieve the pathway, and the score

For the pair *(THT gene, caffeoyltyramine feature)* and the pathway **hydroxycinnamic acid tyramine amides biosynthesis**:

In [25]:
W = pd.read_csv('weights.csv').drop_duplicates(['scope', 'family', 'tier'])
def w(tier, scope):
    s = W[(W.tier == tier) & (W.scope == scope)]; return float(s.weight.iloc[0]) if len(s) else None
tiers = [
 # side, tier, reaches the pathway, why, family, draft weight
 ('gene',       'gene_locus',                                  False, 'the THT genes are not listed in PlantMetWiki',                 'membership',  w('gene-in-plantmetwiki-pathway', 'gene')),
 ('gene',       'gene_uniprot',                                False, 'the pathway lists the decarboxylases, not the THT',            'membership',  w('gene-in-plantmetwiki-pathway', 'gene')),
 ('gene',       'gene_ec (own EC)',                            False, 'no EC number on the tomato entries',                           'membership',  w('gene-in-plantmetwiki-pathway', 'gene')),
 ('gene',       'ortholog (OrthoDB) with EC 2.3.1.110',        True,  'only Solyc08g068710 (the other is not in OrthoDB)',            'sequence',    w('ortholog-in-pathway-species', 'gene')),
 ('gene',       'sequence similarity to EC 2.3.1.110 enzymes', True,  'identity 73-89% for both genes',                               'sequence',    w('ortholog-in-pathway-species', 'gene')),
 ('gene',       'coexpression with another pathway gene',      False, 'the decarboxylase gene correlates (r 0.89) but is about 20-25% identical to the pathway enzymes, so it is not recognised as a pathway gene', 'omics', w('coexpressed-with-pathway-gene', 'gene')),
 ('metabolite', 'inchikey exact / first block',                False, 'caffeoyltyramine is not in PlantMetWiki',                      'identity',    w('inchikey-exact', 'metabolite')),
 ('metabolite', 'structure similarity (atom pairs)',           True,  'the nearest compounds are the pathway amides (with atom pairs, not with bit Morgan)',    'relatedness', w('idsm-similar-metabolites-in-pathway', 'metabolite')),
 ('reaction',   'EC of the reaction (2.3.1.110), Rhea by EC',  True,  'EC is in the pathway; Rhea ids are missing in PlantMetWiki',   'database identifiers', w('rhea', 'reaction')),
 ('reaction',   'Rhea step between features (mass shift)',     False, 'analogy only: no Rhea reaction for the compound',              'omics',       w('rhea-step-between-metabolites', 'reaction')),
]
T = pd.DataFrame(tiers, columns=['side', 'tier', 'reaches the pathway', 'why', 'family', 'draft weight']); T

,side,tier,reaches the pathway,why,family,draft weight
0,gene,gene_locus,False,the THT genes are not listed in PlantMetWiki,membership,0.90
1,gene,gene_uniprot,False,"the pathway lists the decarboxylases, not the THT",membership,0.90
2,gene,gene_ec (own EC),False,no EC number on the tomato entries,membership,0.90
3,gene,ortholog (OrthoDB) with EC 2.3.1.110,True,only Solyc08g068710 (the other is not in OrthoDB),sequence,0.50
4,gene,sequence similarity to EC 2.3.1.110 enzymes,True,identity 73-89% for both genes,sequence,0.50
5,gene,coexpression with another pathway gene,False,the decarboxylase gene correlates (r 0.89) but is about 20-25% ide...,omics,0.50
6,metabolite,inchikey exact / first block,False,caffeoyltyramine is not in PlantMetWiki,identity,0.95
7,metabolite,structure similarity (atom pairs),True,"the nearest compounds are the pathway amides (with atom pairs, not...",relatedness,0.50
8,reaction,"EC of the reaction (2.3.1.110), Rhea by EC",True,EC is in the pathway; Rhea ids are missing in PlantMetWiki,database identifiers,0.60
9,reaction,Rhea step between features (mass shift),False,analogy only: no Rhea reaction for the compound,omics,0.60


In [26]:
def noisy_or(fams): return 1 - np.prod([1 - x for x in fams])
fam = T[T['reaches the pathway']].groupby(['side', 'family'])['draft weight'].max().reset_index(); print(fam.to_string(index=False))
g_seq = fam[fam.family == 'sequence']['draft weight'].max(); m_sim = fam[fam.side == 'metabolite']['draft weight'].max(); r_db = fam[fam.family == 'database identifiers']['draft weight'].max()
variants = {
 'a. the families that fire, as independent (draft model)': [g_seq, m_sim, r_db],
 'b. the EC counted once (sequence tier and reaction tier rest on the same EC 2.3.1.110)': [m_sim, max(g_seq, r_db)],
 'c. as b, without the similarity tier (the fingerprint was chosen knowing the answer)': [max(g_seq, r_db)],
}
pd.DataFrame({'score': {k: round(noisy_or(v), 3) for k, v in variants.items()}})

      side               family  draft weight
      gene             sequence           0.5
metabolite          relatedness           0.5
  reaction database identifiers           0.6


,score
"a. the families that fire, as independent (draft model)",0.9
b. the EC counted once (sequence tier and reaction tier rest on the same EC 2.3.1.110),0.8
"c. as b, without the similarity tier (the fingerprint was chosen knowing the answer)",0.6


In [27]:
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
order = {'gene': 0, 'metabolite': 1, 'reaction': 2}
fig, ax = plt.subplots(figsize=(11, 5.6)); N = max(sum(T.side == k) for k in order); ax.set_xlim(0, 3); ax.set_ylim(-1.3, N + 1.0); ax.axis('off')
for side, x in order.items():
    sub = T[T.side == side].reset_index(drop=True)
    ax.text(x + 0.5, N + 0.45, {'gene': 'GENE (THT)', 'metabolite': 'METABOLITE (caffeoyltyramine)', 'reaction': 'REACTION (THT step)'}[side], ha='center', fontsize=10, fontweight='bold')
    for i, r in sub.iterrows():
        y = N - i - 0.5; ok = r['reaches the pathway']
        ax.add_patch(plt.Rectangle((x + 0.03, y - 0.42), 0.94, 0.84, fc='#cfe8cf' if ok else '#f3d4d0', ec='#555', lw=0.8))
        ax.text(x + 0.5, y + 0.1, r.tier, ha='center', va='center', fontsize=7.2, fontweight='bold')
        ax.text(x + 0.5, y - 0.2, ('reaches the pathway' if ok else 'does not') + f"  (w {r['draft weight']:.2f})", ha='center', va='center', fontsize=6.8, color='#333')
ax.legend(handles=[Patch(fc='#cfe8cf', ec='#555', label='the tier reaches the pathway'), Patch(fc='#f3d4d0', ec='#555', label='the tier does not (reasons in the table above)')], loc='lower center', ncol=2, fontsize=8, frameon=False, bbox_to_anchor=(0.5, 0.0))
fig.suptitle('At which tier is "hydroxycinnamic acid tyramine amides biosynthesis" retrieved for the THT-caffeoyltyramine association?', fontsize=10)
fig.savefig('figures/tht_tier_ladder.png', dpi=170, bbox_inches='tight'); plt.show()

In [28]:
order = [('gene by sequence', g_seq), ('metabolite by structure', m_sim), ('reaction by EC', r_db)]
cum, run = [], 0.0
for _, w_ in order: run = 1 - (1 - run) * (1 - w_); cum.append(run)
fig, (a, b) = plt.subplots(1, 2, figsize=(11.5, 3.9), gridspec_kw={'width_ratios': [1.25, 1]})
xs = np.arange(len(order) + 1)
a.bar(xs, [0] + cum, color=[GREY] + [BLUE] * len(order), width=0.6)
labels = ['direct tiers only\n(pilot)'] + [f'+ {n}\n(w {w_:.2f})' for n, w_ in order]
for x_, v in zip(xs, [0] + cum): a.text(x_, v + 0.02, f'{v:.2f}', ha='center', fontsize=9)
a.set_xticks(xs); a.set_xticklabels(labels, fontsize=8.2); a.set_ylim(0, 1.05); a.set_ylabel('score = 1 - prod(1 - w)'); a.set_title('a  Each family that fires raises the score', loc='left', fontsize=9.5, fontweight='bold')
vn = ['a  as independent', 'b  EC counted once', 'c  b, without\nthe similarity tier']; vv = [noisy_or(v) for v in variants.values()]
b.bar(range(3), vv, color=[BLUE, BLUE, BLUE], width=0.55, alpha=[1.0, 0.75, 0.5][0])
for i, (v, al) in enumerate(zip(vv, (1.0, 0.75, 0.5))): b.patches[i].set_alpha(al); b.text(i, v + 0.02, f'{v:.2f}', ha='center', fontsize=9)
b.set_xticks(range(3)); b.set_xticklabels(vn, fontsize=8.2); b.set_ylim(0, 1.05); b.set_title('b  The score depends on what counts as independent', loc='left', fontsize=9.5, fontweight='bold')
fig.tight_layout(); save(fig, 'tht_score'); plt.show()

**Reading 6.** With the draft weights the three families that fire (gene by sequence, metabolite by structure, reaction by EC) give a score of **0.90**, against **0** from the direct tiers of the pilot. That number depends on what we call independent: the sequence tier and the reaction tier **rest on the same EC number 2.3.1.110**, so counting that EC once gives **0.80**, and without the similarity tier (whose fingerprint was chosen knowing the answer) **0.60**. The range 0.6-0.9 is the honest answer; narrowing it is the job of the calibration planned in [`scoring.md`](../scoring.md), not of this example. What the example shows is the **pattern**: no direct tier fires, several indirect tiers fire from different sides (gene, metabolite, reaction), each indirect tier has a documented way to fail, and the strongest observation of all (the decarboxylase correlating at 0.89) is not counted because it cannot be linked by a rule.

## From one association to the whole dataset

The example is one gene and one compound. How far does the same chain go for **all** genes of the MEANtools associations? The figure counts the genes (from the graphs and from the results of notebooks 02 and 05) at each level of linking.

In [29]:
RES = Path('results'); GEN = PILOT / 'results' / 'gene_annotation.ttl'
assoc_genes = set(re.findall(r'Gene/Summary\?g=(Solyc\d+g\d+)', (Path('../../shape_constraints/rdf/association.ttl')).read_text()))
ga = GEN.read_text(); blocks = ga.split(' .\n')
n_uni = sum('lodp:uniprot' in b for b in blocks); n_ec = sum('lodp:ecNumber' in b for b in blocks); n_orth = sum('lodp:orthologOf' in b for b in blocks)
rg = pd.read_csv(RES / 'ranked-pathway-genes.csv'); rg['gene'] = rg.component.str.extract(r'(Solyc\d+g\d+)')[0]; rg = rg.dropna(subset=['gene'])
tp = rg.tiersPassed.fillna('')
def genes_with(t): return rg[tp.str.contains(t)].gene.nunique()
try: pairs = len(pd.read_csv(RES / 'gene-and-metabolite-in-same-pathway.csv'))
except Exception: pairs = 0
steps = [('genes with a MEANtools association', len(assoc_genes), GREY, 'data'), ('with a UniProt entry', n_uni, GREY, 'link'), ('with an EC number', n_ec, GREY, 'link'),
         ('reach a pathway: own EC number (weak, 0.40)', genes_with('gene-ec-in-pathway'), BLUE, 'tier'), ('reach a pathway: ortholog of a pathway enzyme (0.50)', genes_with('ortholog'), BLUE, 'tier'),
         ('reach a pathway: UniProt entry listed (0.80)', genes_with('gene-uniprot'), BLUE, 'tier'), ('reach a pathway: Solyc locus listed (0.90)', genes_with('gene-in-plantmetwiki'), BLUE, 'tier'),
         ('gene AND annotated metabolite in the same pathway', pairs, ORANGE, 'tier')]
fig, ax = plt.subplots(figsize=(10.5, 4.6)); y = np.arange(len(steps))[::-1]
ax.barh(y, [s_[1] for s_ in steps], color=[s_[2] for s_ in steps], height=0.62)
for yy, s_ in zip(y, steps): ax.text(max(s_[1], 0) + 5, yy, str(s_[1]), va='center', fontsize=9)
ax.set_yticks(y); ax.set_yticklabels([s_[0] for s_ in steps], fontsize=8.6); ax.set_xlim(0, max(s_[1] for s_ in steps) * 1.12); ax.set_xlabel('number of genes (tomato)')
ax.legend(handles=[Patch(fc=GREY, label='annotation available in the graph'), Patch(fc=BLUE, label='gene reaches a PlantMetWiki pathway'), Patch(fc=ORANGE, label='combined gene + metabolite tier')], loc='lower right', frameon=False, fontsize=8)
ax.set_title('Each added tier connects more genes to the knowledge graph; the strict combined tier still finds none', loc='left', fontsize=10, fontweight='bold')
fig.tight_layout(); save(fig, 'tht_funnel'); plt.show()
print([(s_[0], s_[1]) for s_ in steps])

[('genes with a MEANtools association', 350), ('with a UniProt entry', 315), ('with an EC number', 98), ('reach a pathway: own EC number (weak, 0.40)', 47), ('reach a pathway: ortholog of a pathway enzyme (0.50)', 23), ('reach a pathway: UniProt entry listed (0.80)', 0), ('reach a pathway: Solyc locus listed (0.90)', 1), ('gene AND annotated metabolite in the same pathway', 0)]


### The same funnel for all correlated genes, with the sequence tier, and where the THT genes fall

The funnel above counts the 350 genes of the clustered associations in the evidence graph. The THT genes are **not** among them (their clusters are not significant), so they cannot appear there. The table and figure below use **all 5,380 genes** of the correlation table and add the **sequence-similarity tier** (an EC number transferred from proteins in the same UniRef50 cluster, that is at least 50 % identical; `pilots/pairedomics_overlap/sequence_transfer.py`). The last two columns show the two THT genes. The last row is the strict combined tier (gene and an annotated correlated feature in the same pathway; MR <= 10, r > 0), read against a null that shuffles the pathway annotation among the genes.

In [30]:
ft = pd.read_csv(PILOT / 'results' / 'funnel_all_genes.csv'); import json as _json
nl = _json.load(open(PILOT / 'results' / 'funnel_combined_null.json'))
display(ft.rename(columns={'tier': 'step', 'clustered_350': 'clustered genes (350)', 'all_5380': 'all correlated genes (5,380)'}).set_index('step'))
print(f"combined tier: {nl['observed_genes']} genes ({nl['observed_pairs']} gene-feature pairs) against {nl['null_mean']:.1f} expected from a shuffled annotation (95th percentile {nl['null_p95']:.0f}; p = {nl['p_ge_observed']:.3f})")
kind = ['data', 'link', 'link', 'link', 'tier', 'tier', 'tier', 'tier', 'tier', 'tier', 'comb']
col = {'data': GREY, 'link': GREY, 'tier': BLUE, 'comb': ORANGE}
fig, ax = plt.subplots(figsize=(12.4, 5.6)); y = np.arange(len(ft))[::-1]
ax.barh(y, ft.all_5380, color=[col[k] for k in kind], height=0.62, label='all correlated genes')
ax.barh(y, ft.clustered_350, color='none', edgecolor=INK, hatch='////', height=0.62, lw=0.6, label='of which in the clustered associations (350)')
for yy, v in zip(y, ft.all_5380): ax.text(v + 60, yy, f'{v:,}', va='center', fontsize=9)
ax.set_yticks(y); ax.set_yticklabels(ft.tier, fontsize=8.4); ax.set_xlim(0, ft.all_5380.max() * 1.42); ax.set_xlabel('number of genes (tomato)')
xr = ft.all_5380.max() * 1.22
for dx, g in ((0, 'Solyc08g068730'), (ft.all_5380.max() * 0.14, 'Solyc08g068710')):
    ax.text(xr + dx, len(ft) - 0.35, 'THT\n…' + g[-6:], ha='center', va='bottom', fontsize=7.8, fontweight='bold', color=INK)
    for yy, v in zip(y, ft[g]): ax.text(xr + dx, yy, 'yes' if v == 'yes' else '-', ha='center', va='center', fontsize=9, color=GREEN if v == 'yes' else MUTED, fontweight='bold' if v == 'yes' else 'normal')
ax.legend(handles=[Patch(fc=GREY, label='annotation available'), Patch(fc=BLUE, label='gene reaches a PlantMetWiki pathway'), Patch(fc=ORANGE, label='combined gene + metabolite tier'), Patch(fc='none', ec=INK, hatch='////', label='hatched: the 350 clustered genes')], loc='lower right', bbox_to_anchor=(0.66, 0.0), frameon=False, fontsize=8)
ax.set_title('Outside the significant clusters the sequence tier reaches the THT genes; the direct tiers still do not', loc='left', fontsize=10, fontweight='bold', pad=36)
ax.spines['left'].set_visible(False); fig.tight_layout(); save(fig, 'tht_funnel_all'); plt.show()

,clustered genes (350),"all correlated genes (5,380)",Solyc08g068730,Solyc08g068710
step,,,,
genes with a MEANtools correlation/association,350,5380,yes,yes
with a UniProt entry,315,4776,yes,yes
with an EC number of its own,98,1035,no,no
with an EC number transferred from a similar protein (UniRef50),124,1457,yes,yes
reach a pathway: own EC (0.40),45,409,no,no
"reach a pathway: transferred EC, sequence similarity (0.50)",61,688,yes,yes
reach a pathway: ortholog of a pathway enzyme (0.50),22,238,no,no
reach a pathway: UniProt entry listed (0.80),0,2,no,no
reach a pathway: Solyc locus listed (0.90),0,16,no,no


combined tier: 7 genes (8 gene-feature pairs) against 1.9 expected from a shuffled annotation (95th percentile 4; p = 0.003)


## 7. What this example teaches about reading MEANtools output

1. **A correlation is a lead, not a pathway.** r = 0.88 and MR = 7.6 say that two signals move together under the treatment; 107 genes did the same for this feature. The tiers decide whether the movement is *biochemically explainable*; here it is (a transferase and the amide it makes).
2. **Read the pair first, the cluster second.** The cluster p-values of these genes are about 1; the pair is a strong lead. A significant cluster can be a respiration module (the 51-gene cluster that contains sinapoyltyramine in the pilot is mostly mitochondrial and housekeeping genes), and a non-significant cluster can hold a real enzyme-product pair.
3. **Count compounds, not peaks.** Adducts and isotopes of one compound are several features and several "associations" (`M300T887`, `M301T887`, `M322T887` are one compound). Merge them before counting and before mapping by mass, or the isotope gets a false identity (salidroside, guanosine).
4. **Zero overlap with PlantMetWiki is mostly an annotation gap, not a negative result.** The pilot's 0 of 721 comes from gaps seen here: the pathway lists the THT step without a UniProt enzyme or a Rhea id, the tomato entries have no EC number, and one compound is absent. Each gap has a tier that bridges it (sequence or orthology, structure similarity, EC-to-Rhea).
5. **Agreement from different sides is the evidence, if the sides are independent.** The gene reaches the pathway by sequence, the compound by structure, the reaction by EC. Any one alone is a weak hint. But the gene and the reaction tiers share the same EC number, so they are not independent (section 6). Also **a strong correlation can be missed by the rules**: the decarboxylase correlates at 0.89 but cannot be linked to the pathway by sequence.
6. **Report why a tier fails.** OrthoDB lacks one gene, Rhea lacks the substrate, the pathway lists a different enzyme, the decarboxylase is too distant by sequence, Morgan fingerprints rank chalcones first. A failure is a statement about a database or a method, not about the biology.
7. **Pathway numbers change between releases** (PC712 in the pilot snapshot, PC536 in r17); join on the PlantCyc id.
8. **Next steps for this lead.** Confirm the compound (MS/MS or a standard), check the induction of the THT genes in the original RNA-seq, and, if it holds, propose caffeoyl-CoA + tyramine as a reaction to PlantMetWiki and Rhea.

**Limits.** One compound and one pathway, identified at level 3-4, with hand-written structures; the example *illustrates* the tiers and does **not** estimate how often they succeed (that is the calibration). The fingerprint, the identity threshold and the weights are drafts, and the fingerprint was compared knowing the answer.

## A reading guide as a diagram

The sequence of questions that this example follows, for any association that MEANtools reports.

In [31]:
fig, ax = plt.subplots(figsize=(11.5, 7.0)); ax.set_xlim(0, 11.5); ax.set_ylim(0, 7.0); ax.axis('off')
def box(x, y, w, h, t, fc, ec, fs=8.8, bold=False):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.02,rounding_size=0.12', fc=fc, ec=ec, lw=1.3)); ax.text(x + w / 2, y + h / 2, t, ha='center', va='center', fontsize=fs, fontweight='bold' if bold else 'normal')
def arr(x1, y1, x2, y2, t='', c=MUTED):
    ax.add_patch(FancyArrowPatch((x1, y1), (x2, y2), arrowstyle='-|>', mutation_scale=12, color=c, lw=1.5, shrinkA=1, shrinkB=1))
    if t: ax.text((x1 + x2) / 2 + 0.08, (y1 + y2) / 2, t, fontsize=7.8, color=c, ha='left', va='center', bbox=dict(fc='white', ec='none', pad=0.6))
box(3.1, 6.2, 5.3, 0.65, '1  A MEANtools pair: gene, feature, r, MR (and the cluster p-value)', '#dbe9f4', BLUE, bold=True)
box(3.1, 5.0, 5.3, 0.8, '2  Which compound? Merge adducts and isotopes of one compound,\nuse retention time, MS/MS if available', '#dbe9f4', BLUE)
arr(5.75, 6.2, 5.75, 5.8)
box(0.2, 3.1, 3.4, 1.35, '3a  Gene to pathway\nlocus, UniProt entry, EC number,\northology, sequence similarity,\nco-expression with a pathway gene', '#dff0e4', GREEN)
box(4.05, 3.1, 3.4, 1.35, '3b  Metabolite to pathway\nInChIKey (exact, first block),\nstructure similarity (fingerprint!),\nchemical class', '#dff0e4', GREEN)
box(7.9, 3.1, 3.4, 1.35, '3c  Reaction\nEC number, Rhea (join by EC),\nmass shifts between features', '#dff0e4', GREEN)
for xc in (1.9, 5.75, 9.6): arr(5.75, 5.0, xc, 4.45)
box(2.6, 1.7, 6.3, 0.9, '4  Do independent routes lead to the same pathway?\nAre they really independent (shared EC, shared sequence evidence)?', '#fbead7', ORANGE, bold=True)
for xc in (1.9, 5.75, 9.6): arr(xc, 3.1, 5.75, 2.6)
box(0.2, 0.15, 5.1, 0.95, 'yes: a lead for that pathway; rank it with the score\nand report the range under the dependence assumptions', '#dff0e4', GREEN)
box(6.2, 0.15, 5.1, 0.95, 'no: keep it as a lead without a pathway; say why each tier\nfailed (annotation gap, database coverage, new chemistry)', '#f3f4f6', MUTED)
arr(4.3, 1.7, 2.75, 1.1, 'yes', GREEN); arr(7.2, 1.7, 8.75, 1.1, 'no', MUTED)
ax.text(11.4, 0.0, 'then: validate (MS/MS or standard, knockdown), and add the new reaction to the KG', ha='right', fontsize=8, color=MUTED)
fig.suptitle('How to read a MEANtools association with the knowledge graph', fontsize=11.5, y=0.97)
save(fig, 'tht_how_to_read'); plt.show()